# Fine-tuning, probes and figures (thin wrapper)
Attach as inputs: the code dataset (or set `REPO_URL`) and the **output of every earlier session**
(pretrain checkpoints and earlier `results/runs.csv`, which are merged automatically so LR choices carry over).
Settings: GPU T4, Internet On.

| Session | `BLOCKS` | Needs |
| --- | --- | --- |
| Day 2 gate (quick) | `probe_ep100` | `encoder_m75_ep100.pt` (exists from epoch 100 on) |
| Day 2 | `ft_low` | MAE-75 at 400 epochs |
| Day 3 | `ft_100` | MAE-75 at 400 epochs |
| Day 3 | `ft_ablation probes figures` | MAE-50/90 pretrains + the `ft_low` output |

If time runs short at 100%: set `LRS_100 = "5e-4 1e-3"` (design doc risk table).

In [ ]:
BLOCKS = "ft_low"
LRS_100 = ""   # e.g. "5e-4 1e-3" to sweep only two LRs at 100%

In [ ]:
# --- Code: attach this repo as a Kaggle Dataset named "mae-code" (zip of the repo), OR set REPO_URL.
import glob, os, shutil, subprocess
REPO_URL = ""   # e.g. "https://github.com/<you>/mae-label-efficiency.git"
CODE = "/tmp/mae"   # outside /kaggle/working so code + smoke files are not saved as notebook output
shutil.rmtree(CODE, ignore_errors=True)
hits = glob.glob("/kaggle/input/**/scripts/run_block.sh", recursive=True)
if hits:                                   # repo attached as a dataset (any folder depth)
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE)
elif REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=True)
else:
    raise SystemExit("Attach the code dataset or set REPO_URL")
os.chdir(CODE)
os.environ.update(DATA="/kaggle/working/data", RESULTS="/kaggle/working/results",
                  CKPT="/kaggle/working/checkpoints",
                  SEARCH="/kaggle/working/checkpoints /kaggle/input")
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch, torchvision; print(torch.__version__, torchvision.__version__, torch.cuda.is_available())"

In [ ]:
if LRS_100: os.environ["LRS_100"] = LRS_100
!bash scripts/run_block.sh {BLOCKS}

In [ ]:
!tail -n 30 /kaggle/working/results/runs.csv
!ls /kaggle/working/results/figures /kaggle/working/results/tables 2>/dev/null